# **Physics Generator — Design Philosophy**

The physics generator creates synthetic missile engagement scenarios for ATAS.

Real engagement data is difficult and unrealistic to obtain at scale, so synthetic data is used to generate controlled training scenarios with known ground-truth labels.

The generator uses metadata-derived aircraft capability ranges from `aircraft_metadata.csv` instead of fully hardcoded values.


### **aircraft_metadata.csv**

This CSV is the threat database for ATAS.

The classifier only predicts:
```python
"F22"
````

The metadata adds tactical information:

* missile speed
* missile range
* aircraft generation
* maneuverability
* combat capability

This allows the system to simulate engagement scenarios.

---

### Columns

#### `aircraft`

Aircraft class name.

Used to match classifier output with metadata.

---

#### `missile_speed`

Approx missile speed (m/s).

Used for:

* closure rate
* evasion time
* threat level

Higher speed = less reaction time.

---

#### `missile_range`

Approx max missile range (m).

Used for:

* launch distance generation
* engagement realism

Higher range = longer reach.

---

#### `enemy_generation`

Aircraft technology level.

Values:

* 3.5
* 4
* 4.5
* 5

Used for:

* threat weighting
* hit probability modifiers

Higher generation = more dangerous.

---

#### `maneuverability`

Aircraft agility.

Values:

* 0 = low
* 1 = medium
* 2 = high

Used for:

* evasion logic
* survival probability

Higher maneuverability = harder to hit.

---

#### `no_aa_capability`

Whether aircraft lacks air-to-air combat capability.

Values:

* 0 = combat capable
* 1 = not combat capable

Used to:

* lower threat score
* skip missile logic for support aircraft

---

## Why This Exists

The metadata converts:

```python
"What aircraft is this?"
```

into:

```python
"How dangerous is this aircraft?"
```

In [67]:
# Importing the libraries

import pandas as pd
import numpy as np

In [68]:
# Importing the CSV to work with

df = pd.read_csv("../data/aircraft_metadata.csv")
df

,aircraft,aircraft_max_speed,missile_speed,missile_range,enemy_generation,maneuverability,no_aa_capability
0,A10,222,857,35000,4.0,1,0
1,A400M,255,-1,-1,4.0,0,1
2,AG600,155,-1,-1,4.0,0,1
3,AH64,101,750,8000,4.0,1,0
4,AKINCI,100,1372,65000,4.0,1,0
...,...,...,...,...,...,...,...
97,Y20,255,-1,-1,4.0,0,1
98,YF23,648,1372,160000,5.0,2,0
99,Z10,83,686,8000,4.0,1,0
100,Z19,78,686,8000,4.0,1,0


In [69]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   aircraft            102 non-null    str    
 1   aircraft_max_speed  102 non-null    int64  
 2   missile_speed       102 non-null    int64  
 3   missile_range       102 non-null    int64  
 4   enemy_generation    102 non-null    float64
 5   maneuverability     102 non-null    int64  
 6   no_aa_capability    102 non-null    int64  
dtypes: float64(1), int64(5), str(1)
memory usage: 5.7 KB


In [70]:
df.describe(include="all")

,aircraft,aircraft_max_speed,missile_speed,missile_range,enemy_generation,maneuverability,no_aa_capability
count,102,102.000000,102.000000,102.000000,102.000000,102.000000,102.000000
unique,102,NaN,NaN,NaN,NaN,NaN,NaN
top,A10,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,345.764706,636.460784,64724.990196,4.117647,0.852941,0.500000
std,NaN,226.619242,692.537831,102267.602554,0.478118,0.825306,0.502469
min,NaN,61.000000,-1.000000,-1.000000,3.500000,0.000000,0.000000
25%,NaN,164.500000,-1.000000,-1.000000,4.000000,0.000000,0.000000
50%,NaN,268.000000,342.500000,3999.500000,4.000000,1.000000,0.500000
75%,NaN,544.000000,1372.000000,108750.000000,4.500000,2.000000,1.000000


In [71]:
print(df["enemy_generation"].unique())
print(df["maneuverability"].unique())
print(df["no_aa_capability"].unique())

[4.  3.5 4.5 5. ]
[1 0 2]
[0 1]


In [72]:
# Getting data of aircarfts that has air-to-air combact ability
combat_df = df[df["no_aa_capability"]==0].reset_index(drop=True)
combat_df.head()

,aircraft,aircraft_max_speed,missile_speed,missile_range,enemy_generation,maneuverability,no_aa_capability
0,A10,222,857,35000,4.0,1,0
1,AH64,101,750,8000,4.0,1,0
2,AKINCI,100,1372,65000,4.0,1,0
3,AV8B,300,1372,160000,4.0,1,0
4,EF2000,590,1372,200000,4.5,2,0


In [73]:
combat_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   aircraft            51 non-null     str    
 1   aircraft_max_speed  51 non-null     int64  
 2   missile_speed       51 non-null     int64  
 3   missile_range       51 non-null     int64  
 4   enemy_generation    51 non-null     float64
 5   maneuverability     51 non-null     int64  
 6   no_aa_capability    51 non-null     int64  
dtypes: float64(1), int64(5), str(1)
memory usage: 2.9 KB


In [74]:
combat_df.describe(include="all")

,aircraft,aircraft_max_speed,missile_speed,missile_range,enemy_generation,maneuverability,no_aa_capability
count,51,51.000000,51.000000,51.000000,51.000000,51.000000,51.0
unique,51,NaN,NaN,NaN,NaN,NaN,NaN
top,A10,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,440.235294,1273.921569,129450.980392,4.284314,1.529412,0.0
std,NaN,225.509609,373.955497,112160.655085,0.461030,0.504101,0.0
min,NaN,78.000000,686.000000,8000.000000,3.500000,1.000000,0.0
25%,NaN,246.500000,857.000000,37500.000000,4.000000,1.000000,0.0
50%,NaN,532.000000,1372.000000,110000.000000,4.000000,2.000000,0.0
75%,NaN,590.000000,1372.000000,160000.000000,4.500000,2.000000,0.0


In [75]:
# Getting missile speed ranges
print("Missile Speed ranges: \n")
print(f"Max → {combat_df['missile_speed'].max()} \nMin → {combat_df['missile_speed'].min()}")

Missile Speed ranges: 

Max → 2058 
Min → 686


In [76]:
# Getting missile speed ranges
print("Missile range: \n")
print(f"Max → {combat_df['missile_range'].max()} \nMin → {combat_df['missile_range'].min()}")

Missile range: 

Max → 400000 
Min → 8000


In [77]:
# Getting aircraft speed ranges
print("Aircraft speed range: \n")
print(f"Max → {df['aircraft_max_speed'].max()} \nMin → {df['aircraft_max_speed'].min()}")

Aircraft speed range: 

Max → 983 
Min → 61


In [78]:
# Getting the number of unique generation
combat_df["enemy_generation"].value_counts()

enemy_generation
4.0    25
4.5    11
5.0    11
3.5     4
Name: count, dtype: int64

In [79]:
# Getting the maneverability level
combat_df["maneuverability"].value_counts()

maneuverability
2    27
1    24
Name: count, dtype: int64

In [80]:
# get values of ranges of missile speed
combat_df.sort_values("missile_speed", ascending=False).head(10)

,aircraft,aircraft_max_speed,missile_speed,missile_range,enemy_generation,maneuverability,no_aa_capability
14,FCK1,532,2058,100000,4.0,2,0
39,Su57,590,2058,400000,5.0,2,0
32,Mig31,833,2058,400000,4.0,1,0
18,J36,590,1715,400000,5.0,2,0
19,J50,590,1715,400000,5.0,2,0
17,J35,590,1715,300000,5.0,2,0
16,J20,590,1715,300000,5.0,2,0
15,J10,650,1715,300000,4.5,2,0
10,F2,590,1544,105000,4.5,2,0
41,Tejas,550,1544,110000,4.5,2,0


In [81]:
# Get sorted decending missile ranges
combat_df.sort_values("missile_range", ascending=False)[:10]

,aircraft,aircraft_max_speed,missile_speed,missile_range,enemy_generation,maneuverability,no_aa_capability
19,J50,590,1715,400000,5.0,2,0
39,Su57,590,2058,400000,5.0,2,0
18,J36,590,1715,400000,5.0,2,0
32,Mig31,833,2058,400000,4.0,1,0
17,J35,590,1715,300000,5.0,2,0
16,J20,590,1715,300000,5.0,2,0
15,J10,650,1715,300000,4.5,2,0
34,Rafale,531,1372,200000,4.5,2,0
20,JAS39,590,1372,200000,4.5,2,0
24,KF21,532,1372,200000,4.5,2,0


In [82]:
combat_df[combat_df["aircraft"]=="F22"]

,aircraft,aircraft_max_speed,missile_speed,missile_range,enemy_generation,maneuverability,no_aa_capability
11,F22,669,1372,160000,5.0,2,0


In [83]:
# correlation between aircraft speed and missile 
combat_df[["aircraft_max_speed", "missile_speed", "missile_range"]].corr()

,aircraft_max_speed,missile_speed,missile_range
aircraft_max_speed,1.000000,0.753865,0.633663
missile_speed,0.753865,1.000000,0.820434
missile_range,0.633663,0.820434,1.000000


In [84]:
combat_df.groupby("enemy_generation")["missile_range"].mean()

enemy_generation
3.5     26500.000000
4.0     83240.000000
4.5    159090.909091
5.0    242272.727273
Name: missile_range, dtype: float64

## **Extracted Metadata Insights**

### **Combat-Capable Aircraft**
- Total aircraft: 102
- Combat-capable aircraft: 56
- Non combat-capable aircraft: 46

---

### Aircraft Speed Range

```python
61 m/s → 983 m/s
```

---

### Missile Speed Range

```python 
686 m/s → 2058 m/s
````

---

### Missile Range

```python id="e2h15d"
8,000 m → 400,000 m
```

---

### Enemy Generation Distribution

```python id="txut2q"
4.0  → 23 aircraft
5.0  → 13 aircraft
4.5  → 10 aircraft
3.5  → 5 aircraft
```

---

### Maneuverability Distribution

```python id="nt11ao"
2 (high)   → 27 aircraft
1 (medium) → 24 aircraft
```

---

### Key Observations

* Most combat aircraft belong to Gen 4 and Gen 5.
* Most combat aircraft have medium or high maneuverability.
* Advanced aircraft generally have longer missile ranges.
* High-end aircraft like Su57, Mig31, J35, J36, and J50 dominate the upper missile range limits.

The goal of the generator is not perfect aerospace simulation, but generation of believable and internally consistent combat scenarios that allow ML models to learn meaningful relationships.

---

## **Plan to build physics generator**

### **How it should be called**

```python
from src.physics_generator import generate_dataset
generate_dataset()
```

### **How it will be build**

```markdown
generate_dataset()          ← the one function the notebook calls
    └── load_metadata()     ← reads aircraft_metadata.csv
    └── generate_row()      ← builds one scenario
            └── derive_missile_phase()
            └── derive_closure_rate()
            └── derive_evasion_time()
            └── derive_hit_label()
    └── save_dataset()      ← saves to CSV
```
---

### **The sequence of `physics_generator.py` script**

```python
_generate_row()
│
├── 1. Pick a random enemy aircraft from metadata
│      (combat-capable only — no_aa_capability = 0)
│
├── 2. Sample raw feature values
│      your_speed, your_altitude, azimuth, elevation,
│      maneuverability, countermeasure_deployed
│      launch_distance (within that aircraft's missile_range)
│      remaining_distance (between 0 and launch_distance)
│
├── 3. _derive_missile_phase()
│      "How far has the missile already traveled?"
│      → returns 0, 1, or 2
│
├── 4. _derive_closure_rate()
│      "How fast is the gap closing in 3D space?"
│      → returns a speed in m/s
│
├── 5. _derive_evasion_time()
│      "How many seconds before it reaches you?"
│      → returns a float in seconds
│
├── 6. _derive_hit_label()
│      "After your evasion attempt, does it hit?"
│      → returns 0 or 1
│
└── 7. Package everything into a dict → one row
```